In [1]:
import csv
import yaml
import os
import subprocess

In [2]:
def csv_to_dict(filename):
    """Converts a CSV file to a list of dictionaries."""
    with open(filename, 'r') as file:
        reader = csv.DictReader(file)
        return list(reader)


In [3]:
def header(name, p, n, mem, time, email):
    """
    PARAMETERS
    ----------
    name: string
        name of batch job
    p: string
        node - options of 'node', 'seseml', 'sesempi', 'sesebig'
    n: string
        Number of processors needed for the job
    mem: string
        Total RAM required: i.e. '64GB'
    time: string
        Time needed for the job: 'Days-HH:MM:SS'
    email: string
        Email where info about execution is sent
    """
    string = '#!/bin/bash\n\n'
    string += '#SBATCH --job-name={}\n'.format(name)
    string += '#SBATCH -p {}\n'.format(p)
    string += '#SBATCH -n {}\n'.format(n)
    string += '#SBATCH --mem {}\n'.format(mem)
    string += '#SBATCH --time={}\n'.format(time)
    string += '#SBATCH --mail-type=BEGIN\n'
    string += '#SBATCH --mail-type=FAIL\n'
    string += '#SBATCH --mail-type=END\n'
    string += '#SBATCH --mail-user={}\n\n'.format(email)
    return string

def runPython(condaEnv, script, args, output=None):
    string = 'source ~/miniconda3/bin/activate {}\n\n'.format(condaEnv)
    string += 'python {}'.format(script)
    for arg in args:
        string += ' {}'.format(arg)
    if(output != None):
        string += ' > {}'.format(output)
    string += '\n'
    return string

def writeBatch(string, outFile):
    with open(outFile, 'w') as text_file:
        text_file.write(string)
        print('Wrote {}'.format(outFile))

# All together

In [4]:
## Load CSV
csv_file = '/data/keeling/a/johntl3/b/LundstromPub2025/automation/job_lists/CAMP2Ex_RF17_V08.csv'
config = csv_to_dict(csv_file)

floats = ['surface_wind_speed', 'a', 'b', 'error_cov_sf']
conda_env = 'at3d_pure_k9'
at3d_script = '/data/keeling/a/johntl3/b/LundstromPub2025/src/main/run_at3d.py'
for d in config:
    ## generate yml configs
    for f in floats:
        d[f] = float(d[f])
    os.makedirs(d['out_path'], exist_ok=True)
    assert len(os.listdir(d['out_path'])) == 0, '{} contains stuff already'.format(d['out_path'])
    with open('{}config.yml'.format(d['out_path']), 'w') as file:
        yaml.dump(d, file, sort_keys=False)
        print('Wrote {}config.yml'.format(d['out_path']))

    ## Write shell scripts
    name = os.path.basename(d['out_path'][0:-1])
    args = ['{}config.yml'.format(d['out_path'])]
    logfile = '{}at3d.log'.format(d['out_path'])
    hdr = header(name, 'node', '4', '64GB', '10-00:00:00', 'johntl3@illinois.edu')
    body = runPython(conda_env, at3d_script, args, logfile)
    shell_file = '{}run_at3d.sh'.format(d['out_path'])
    writeBatch(hdr+body, shell_file)

    ## Launch batch jobs
    os.chdir(d['out_path'])
    result = subprocess.run(['sbatch', 'run_at3d.sh'], capture_output=True, text=True)
    console_output = result.stdout
    print(console_output)

    ## Git commit
    commit_message = '{}: {}'.format(console_output, d['out_path'])
    try:
        subprocess.run(['git', 'add', '/data/keeling/a/johntl3/b/LundstromPub2025/'], check=True)
        subprocess.run(['git', 'commit', '-m', commit_message], check=True)
        print('Committed to git {}'.format(commit_message))
    except subprocess.CalledProcessError as e:
        print(f'Error during git operation: {e}')
    print('-'*145)

Wrote /data/keeling/a/johntl3/b/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default/config.yml
Wrote /data/keeling/a/johntl3/b/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default/run_at3d.sh
Submitted batch job 73206

[main 84d4c29] Submitted batch job 73206 : /data/keeling/a/johntl3/b/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default/
 10 files changed, 112 insertions(+), 40 deletions(-)
 rename experiments/CAMP2Ex_RF17/V07/{default_restart => default_obsolete}/config.yml (80%)
 rename experiments/CAMP2Ex_RF17/V07/{default_restart => default_obsolete}/run_at3d.sh (68%)
 create mode 100644 experiments/CAMP2Ex_RF17/V08/default/config.yml
 create mode 100644 experiments/CAMP2Ex_RF17/V08/default/run_at3d.sh
Committed to git Submitted batch job 73206
: /data/keeling/a/johntl3/b/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default/
-------------------------------------------------------------------------------------------------------------------------------------------------
Wrote 